In [ ]:
import os
from pathlib import Path
from src.utils import *
from src.config import PROCESSED_IMG_DIR, RAW_IMG_DIR
from src.feature_extraction import *
from src.feature_extraction import _create_fixed_box
from src.region_visualization import collect_all_rois, build_mosaic, show_mosaic
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
from src.config import ROOT_DIR
import time
import cv2
import random
from multiprocessing import Pool, cpu_count
from functools import partial
from src.feature_extraction import _extract_features_for_image

In [ ]:
paths = get_all_image_paths(directory = PROCESSED_IMG_DIR)
separated_paths = seprate_processed_files(paths) # Separate all paths that end in .png
processed_paths = separated_paths[0] # These are the processed images (paths only)
masks_paths = separated_paths[1]     # These are the binary masks (paths only)
skeletons_paths = separated_paths[2] # These are the skeletons (paths only)
xml_paths = []                       # And this for the .xml (paths only)
ground_truth_bboxes = []             # And this for the ground truth bboxes
for img_path in processed_paths:
    xml_path = get_xml_path(img_path)
    xml_path = xml_path.replace('_processed.xml', '_bbox.xml') # Fix the ending from _processed.xml to _bbox.xml
    xml_paths.append(xml_path)
    bbox = parse_stenosis_xml(xml_path)
    ground_truth_bboxes.append(bbox)
print('XML files: ', len(xml_paths))
print('Ground truth bounding boxes: ', len(ground_truth_bboxes))

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 0: Build the 704-image subset (11 frames × 64 patients)
# Strategy: for each patient, find their largest serie, then
# select the 11 frames centred around the middle frame.
# ─────────────────────────────────────────────────────────────
from collections import defaultdict

FRAMES_PER_PATIENT = 11  # min-of-maxs determined from dataset analysis
STAT_NAMES  = ['mean', 'var', 'entropy', 'energy', 'kurtosis', 'skewness']
TILE_LABELS = ['tl', 'tr', 'bl', 'br']
ROI_SIZE    = 80
HALF_SIZE   = ROI_SIZE // 2
LABELING_THRESHOLD = 0.50
N_SIZES = 6
N_ORIENTATIONS = 12

# ── Step 1: Group processed paths by patient and serie ───────
# Path structure: .../dataset_processed/<patientID>/<serieID>/<frame>_processed.png
patient_series = defaultdict(lambda: defaultdict(list))

for path in processed_paths:
    p          = Path(path)
    patient_id = p.parts[-3]
    serie_id   = p.parts[-2]
    patient_series[patient_id][serie_id].append(path)

# Sort frames within each serie
for patient_id in patient_series:
    for serie_id in patient_series[patient_id]:
        patient_series[patient_id][serie_id] = sorted(patient_series[patient_id][serie_id])

# ── Step 2: For each patient, find the largest serie ─────────
largest_series = {}   # patient_id → list of sorted frame paths

for patient_id, series in patient_series.items():
    largest_serie_paths = max(series.values(), key=len)
    largest_series[patient_id] = largest_serie_paths

print(f"Patients found : {len(largest_series)}")
for pid, frames in sorted(largest_series.items()):
    print(f"  Patient {pid} → largest serie has {len(frames)} frames")

# ── Step 3: Select 11 frames centred around the middle ───────
subset_processed_paths = []

for patient_id, frame_paths in sorted(largest_series.items()):
    n      = len(frame_paths)
    mid    = n // 2
    half   = FRAMES_PER_PATIENT // 2

    start  = mid - half
    end    = start + FRAMES_PER_PATIENT

    # Clamp to valid range if the serie has exactly 11 or fewer frames
    start  = max(0, start)
    end    = min(n, start + FRAMES_PER_PATIENT)
    start  = max(0, end - FRAMES_PER_PATIENT)   # re-adjust start if end was clamped

    selected = frame_paths[start:end]
    subset_processed_paths.extend(selected)

print(f"\nTotal frames selected : {len(subset_processed_paths)}  (expected {FRAMES_PER_PATIENT} × {len(largest_series)} = {FRAMES_PER_PATIENT * len(largest_series)})")

# ── Step 4: Derive aligned mask, skeleton and GT bbox lists ──
subset_masks_paths         = [get_skeleton_path(p).replace('_skeleton.png', '_mask.png') for p in subset_processed_paths]
subset_skeletons_paths     = [get_skeleton_path(p) for p in subset_processed_paths]
subset_ground_truth_bboxes = [parse_stenosis_xml(get_bbox_xml_path(p)) for p in subset_processed_paths]

print(f"Masks paths     : {len(subset_masks_paths)}")
print(f"Skeleton paths  : {len(subset_skeletons_paths)}")
print(f"GT bbox lists   : {len(subset_ground_truth_bboxes)}")
print(f"Images with annotations : {sum(1 for b in subset_ground_truth_bboxes if len(b) > 0)}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL A: Extract ROI coordinates → roi_coordinates_subset.csv
# Uses the 704-image subset built in Cell 0.
# ─────────────────────────────────────────────────────────────
OUTPUT_DIR_CSV_SUBSET = Path(os.path.join(ROOT_DIR, "notebooks\csv_files\subset_704images"))
OUTPUT_DIR_CSV_SUBSET.mkdir(parents=True, exist_ok=True)

subset_dataset = list(zip(subset_processed_paths, subset_masks_paths,
                          subset_skeletons_paths, subset_ground_truth_bboxes))

rows_coords_subset = []

print(f"Starting coordinate extraction over {len(subset_dataset)} images...")
start = time.time()

for img_path, mask_path, skel_path, gt_boxes in subset_dataset:

    img      = load_image(img_path)
    skeleton = load_image(skel_path)
    if img is None or skeleton is None:
        continue

    img_shape = img.shape[:2]

    p          = Path(img_path)
    patient_id = p.parts[-3]
    serie_id   = p.parts[-2]
    frame_stem = p.stem.replace('_processed', '')
    image_name = f"{patient_id}_{serie_id}_{frame_stem}"

    sampled_boxes  = extract_rois(skeleton, img_shape, roi_size=ROI_SIZE, total_rois=100)
    sampled_labels = label_rois(sampled_boxes, gt_boxes, threshold=LABELING_THRESHOLD)

    all_boxes_and_labels = list(zip(sampled_boxes, sampled_labels))

    for gt_box in gt_boxes:
        cx    = (gt_box['xmin'] + gt_box['xmax']) // 2
        cy    = (gt_box['ymin'] + gt_box['ymax']) // 2
        fixed = _create_fixed_box(cx, cy, HALF_SIZE, img_shape)
        all_boxes_and_labels.append((fixed, 1))

    for roi_idx, (box, lbl) in enumerate(all_boxes_and_labels, start=1):
        rows_coords_subset.append({
            'roi_name'   : f"{image_name}_{roi_idx}",
            'image_name' : image_name,
            'xmin'       : box['xmin'],
            'ymin'       : box['ymin'],
            'xmax'       : box['xmax'],
            'ymax'       : box['ymax'],
            'label'      : lbl,
        })

elapsed = time.time() - start
df_coords_subset = pd.DataFrame(rows_coords_subset)

print(f"\nDone in {elapsed:.1f}s — {len(rows_coords_subset)} ROIs extracted.")
print(f"df_coords_subset shape : {df_coords_subset.shape}")
print(f"Stenosis ROIs          : {df_coords_subset['label'].sum()}")
print(f"Healthy ROIs           : {(df_coords_subset['label'] == 0).sum()}")

coords_csv_path_subset = OUTPUT_DIR_CSV_SUBSET / "roi_coordinates_subset.csv"
df_coords_subset.to_csv(coords_csv_path_subset, index=False)
print(f"\nSaved: {coords_csv_path_subset}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL B: Extract features for each ROI → roi_features_subset.csv
# Parallelised across all available CPU cores.
# Requires df_coords_subset from Cell A to be in memory.
# ─────────────────────────────────────────────────────────────
subset_dataset = list(zip(subset_processed_paths, subset_masks_paths,
                          subset_skeletons_paths, subset_ground_truth_bboxes))

gabor_bank = build_gabor_bank(ksize=31, n_sizes=N_SIZES, n_orientations=N_ORIENTATIONS)
feature_cols = build_feature_column_names(
    n_filters=len(gabor_bank),
    stat_names=STAT_NAMES,
    tile_labels=TILE_LABELS,
)

worker_fn_subset = partial(
    _extract_features_for_image,
    roi_size           = ROI_SIZE,
    half_size          = HALF_SIZE,
    labeling_threshold = LABELING_THRESHOLD,
    gabor_bank         = gabor_bank,
    feature_cols       = feature_cols,
)

n_cores = cpu_count()
print(f"Starting parallel feature extraction over {len(subset_dataset)} images using {n_cores} cores...")
start = time.time()

with Pool(processes=n_cores) as pool:
    results = pool.map(worker_fn_subset, subset_dataset)

rows_features_subset = [row for image_rows in results for row in image_rows]

elapsed = time.time() - start
df_features_subset = pd.DataFrame(rows_features_subset)

print(f"\nDone in {elapsed:.1f}s — {len(rows_features_subset)} ROIs extracted.")
print(f"df_features_subset shape : {df_features_subset.shape}")
print(f"Stenosis ROIs            : {df_features_subset['label'].sum()}")
print(f"Healthy ROIs             : {(df_features_subset['label'] == 0).sum()}")

features_csv_path_subset = OUTPUT_DIR_CSV_SUBSET / "roi_features_subset.csv"
df_features_subset.to_csv(features_csv_path_subset, index=False)
print(f"\nSaved: {features_csv_path_subset}")

## EXPLORATION OF EXTRACTED FEATURES

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPLORATION: Load CSVs and reconstruct feature matrix
# ─────────────────────────────────────────────────────────────
df_features_704 = pd.read_csv(features_csv_path_subset)

y_704        = df_features_704['label'].values
X_704        = df_features_704.drop(columns=['roi_name', 'label']).values
feature_cols_704 = [c for c in df_features_704.columns if c not in ('roi_name', 'label')]

scaler_704   = StandardScaler()
X_scaled_704 = scaler_704.fit_transform(X_704)

print(f"Loaded feature matrix : {X_704.shape}")
print(f"Stenosis ROIs         : {np.sum(y_704)}")
print(f"Healthy ROIs          : {len(y_704) - np.sum(y_704)}")
print(f"Class imbalance       : {round((len(y_704) - np.sum(y_704)) / np.sum(y_704), 2)}× more healthy than stenosis")

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPLORATION: 3D PCA scatter plot
# ─────────────────────────────────────────────────────────────
%matplotlib tk

pca_3d_704   = PCA(n_components=3)
X_pca_3d_704 = pca_3d_704.fit_transform(X_scaled_704)
total_variance_704 = pca_3d_704.explained_variance_ratio_.sum() * 100

fig = plt.figure(figsize=(10, 8))
ax  = fig.add_subplot(111, projection='3d')

ax.scatter(X_pca_3d_704[y_704 == 0, 0], X_pca_3d_704[y_704 == 0, 1], X_pca_3d_704[y_704 == 0, 2],
           alpha=0.4, label='Healthy (0)',  color='#3498db', s=25)
ax.scatter(X_pca_3d_704[y_704 == 1, 0], X_pca_3d_704[y_704 == 1, 1], X_pca_3d_704[y_704 == 1, 2],
           alpha=0.8, label='Stenosis (1)', color='#e74c3c', marker='x', s=45)

ax.set_title(f"3D PCA — 704-Image Subset\n(Total Explained Variance: {total_variance_704:.1f}%)",
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel(f"PC 1 ({pca_3d_704.explained_variance_ratio_[0]*100:.1f}%)", fontsize=10, labelpad=10)
ax.set_ylabel(f"PC 2 ({pca_3d_704.explained_variance_ratio_[1]*100:.1f}%)", fontsize=10, labelpad=10)
ax.set_zlabel(f"PC 3 ({pca_3d_704.explained_variance_ratio_[2]*100:.1f}%)", fontsize=10, labelpad=10)
ax.xaxis.set_pane_color((0.95, 0.95, 0.95, 1.0))
ax.yaxis.set_pane_color((0.95, 0.95, 0.95, 1.0))
ax.zaxis.set_pane_color((0.95, 0.95, 0.95, 1.0))
ax.view_init(elev=25, azim=45)
ax.legend(loc='upper left', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPLORATION: Feature ranking and boxplots
# ─────────────────────────────────────────────────────────────
%matplotlib inline

f_values_704, p_values_704 = f_classif(X_scaled_704, y_704)

n_indeces = 8
top_n_indices_704 = np.argsort(f_values_704)[::-1][:n_indeces]

print(f"\n--- TOP {n_indeces} MOST DISCRIMINATIVE FEATURES — 704-Image Subset ---")
for rank, idx in enumerate(top_n_indices_704, start=1):
    print(f"Rank {rank}: Feature {idx:4d}  '{feature_cols_704[idx]}'  — F-score: {f_values_704[idx]:.2f}  p: {p_values_704[idx]:.2e}")

labels_704 = np.where(y_704 == 0, 'Healthy', 'Stenosis')

fig, axes = plt.subplots(1, n_indeces, figsize=(16, 5))
fig.suptitle(f"Top {n_indeces} Most Discriminative Features — 704-Image Subset", fontsize=16)

for i, feat_idx in enumerate(top_n_indices_704):
    ax = axes[i]
    sns.boxplot(
        x=labels_704,
        y=X_scaled_704[:, feat_idx],
        hue=labels_704,
        order=['Healthy', 'Stenosis'],
        hue_order=['Healthy', 'Stenosis'],
        palette={'Healthy': '#3498db', 'Stenosis': '#e74c3c'},
        legend=False,
        ax=ax,
    )
    ax.set_title(feature_cols_704[feat_idx], fontsize=7, wrap=True)
    ax.set_xlabel("")
    ax.set_ylabel("Standardised Value" if i == 0 else "")

plt.tight_layout()
plt.show()